# Layer-shape comparison: 224×224 vs. 120×120 input (PyTorch)

This notebook holds the PyTorch code used in the slides of Lecture 4 (Convolutional Neural Networks).

It builds a small, fully-convolutional PyTorch network containing

- **2× spatial downsampling** in several places, via `nn.MaxPool2d`.
- **1×1 convolutions used in place of a fully-connected classifier**, so the
  network is fully convolutional (this is what we will build on again in a
  later slide).
- **Global Average Pooling (GAP)** right before the output, which is what
  lets the exact same network accept both a 224×224 and a 120×120 input
  without any architecture change.

In [1]:
import torch
import torch.nn as nn
from collections import OrderedDict

## 1. A minimal `model.summary()` for PyTorch

PyTorch has no built-in equivalent of Keras' `model.summary()`. The
function below builds one using forward hooks: a hook is registered on
every leaf submodule (a module with no children of its own, e.g. `Conv2d`,
`MaxPool2d`), and each hook records that module's output
shape and parameter count the moment it runs during the forward pass. A
single dummy forward pass is then enough to fill in the whole table.

In [2]:
def summarize(model, input_size, batch_size=1, max_layers=None):
    """A minimal PyTorch equivalent of Keras' model.summary().

    Parameters
    ----------
    model : nn.Module
    input_size : tuple
        (channels, height, width) of a single input, e.g. (3, 224, 224).
    max_layers : int or None
        If given, only print/return the first `max_layers` rows (useful
        for very deep networks where you only want to show the first
        10-20 layers, as in the original MobileNet slide).
    """
    rows = []
    handles = []

    def make_hook(name):
        def hook(module, inputs, output):
            n_params = sum(p.numel() for p in module.parameters(recurse=False))
            rows.append((name, module.__class__.__name__, tuple(output.shape), n_params))
        return hook

    # Only leaf modules (no children) - otherwise a container such as
    # `self.features` would also fire its own hook and double-count.
    for name, module in model.named_modules():
        if name and len(list(module.children())) == 0:
            handles.append(module.register_forward_hook(make_hook(name)))

    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.zeros(batch_size, *input_size))
    model.train(was_training)

    for h in handles:
        h.remove()

    if max_layers is not None:
        rows = rows[:max_layers]

    name_w = max(len(r[0]) for r in rows) + 2
    type_w = max(len(r[1]) for r in rows) + 2
    shape_w = max(len(str(r[2])) for r in rows) + 2

    header = f"{'Layer':{name_w}}{'Type':{type_w}}{'Output Shape':{shape_w}}{'Param #'}"
    print(header)
    print('-' * len(header))
    total_params = 0
    for name, ltype, shape, nparams in rows:
        total_params += nparams
        print(f"{name:{name_w}}{ltype:{type_w}}{str(shape):{shape_w}}{nparams:,}")
    print('-' * len(header))
    print(f"Total parameters shown: {total_params:,}")
    return rows

## 2. For comparison: a standard CNN with Flatten + Linear

Before building the fully-convolutional network, here is the "textbook"
alternative from Section 8.1 of the lecture notes: the same convolutional
backbone, ending in `nn.Flatten()` followed by `nn.Linear(...)`. This is
what most CNN classifiers looked like before GAP-based heads became common
(e.g. AlexNet, VGG).

The catch: `nn.Linear` needs a fixed number of input features, and that
number depends on the spatial size coming out of the convolutional
backbone — which in turn depends on the input image size. Below we
hard-code it for a 224×224 input (128 channels × 28 × 28 spatial =
100,352 features, following the same three 2× poolings as the FCN).

In [3]:
class StandardCNN(nn.Module):
    """A 'classic' CNN classifier: convolutional backbone + Flatten + Linear.

    Uses the same convolutional backbone as TinyFCN (below), but ends with
    Flatten + a fully-connected layer whose input size is hard-coded for a
    224x224 input. It will only work for that one input size.
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(OrderedDict([
            ('conv1', nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)),
            ('relu1', nn.ReLU(inplace=True)),
            ('conv2', nn.Conv2d(32, 32, kernel_size=3, padding=1)),
            ('relu2', nn.ReLU(inplace=True)),
            ('pool1', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample

            ('conv3', nn.Conv2d(32, 64, kernel_size=3, padding=1)),
            ('relu3', nn.ReLU(inplace=True)),
            ('conv4', nn.Conv2d(64, 64, kernel_size=3, padding=1)),
            ('relu4', nn.ReLU(inplace=True)),
            ('pool2', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample

            ('conv5', nn.Conv2d(64, 128, kernel_size=3, padding=1)),
            ('relu5', nn.ReLU(inplace=True)),
            ('pool3', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample
        ]))
        # 224 -> 112 -> 56 -> 28 after three 2x poolings, so with 128
        # channels the flattened feature vector has 128*28*28 = 100,352
        # entries -- but ONLY for a 224x224 input.
        self.classifier = nn.Sequential(OrderedDict([
            ('flatten', nn.Flatten()),
            ('fc',      nn.Linear(128 * 28 * 28, num_classes)),
        ]))

    def forward(self, x):
        return self.classifier(self.features(x))

standard_model = StandardCNN(in_channels=3, num_classes=10)

### 2.1  It works at 224×224 — the size it was hard-coded for

In [5]:
summarize(standard_model, input_size=(3, 224, 224));

Layer               Type       Output Shape       Param #
---------------------------------------------------------
features.conv1      Conv2d     (1, 32, 224, 224)  896
features.relu1      ReLU       (1, 32, 224, 224)  0
features.conv2      Conv2d     (1, 32, 224, 224)  9,248
features.relu2      ReLU       (1, 32, 224, 224)  0
features.pool1      MaxPool2d  (1, 32, 112, 112)  0
features.conv3      Conv2d     (1, 64, 112, 112)  18,496
features.relu3      ReLU       (1, 64, 112, 112)  0
features.conv4      Conv2d     (1, 64, 112, 112)  36,928
features.relu4      ReLU       (1, 64, 112, 112)  0
features.pool2      MaxPool2d  (1, 64, 56, 56)    0
features.conv5      Conv2d     (1, 128, 56, 56)   73,856
features.relu5      ReLU       (1, 128, 56, 56)   0
features.pool3      MaxPool2d  (1, 128, 28, 28)   0
classifier.flatten  Flatten    (1, 100352)        0
classifier.fc       Linear     (1, 10)            1,003,530
---------------------------------------------------------
Total parameters 

### 2.2  It breaks at 120×120

At 120×120 the backbone produces a `(128, 15, 15)` feature map, which
flattens to 128×15×15 = 28,800 features — not the 100,352 that `fc`
expects. `nn.Linear` cannot silently adapt, so the forward pass raises a
shape-mismatch error.

In [6]:
try:
    summarize(standard_model, input_size=(3, 120, 120))
except RuntimeError as e:
    print(f"RuntimeError: {e}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x28800 and 100352x10)


This is exactly the limitation the lecture notes describe in Section
8.2: a `Flatten` + `Linear` classifier locks the network to one input
size. The fully-convolutional network below fixes this by replacing that
classifier with a 1×1 convolution + Global Average Pooling, both of which
adapt automatically to any spatial size.

## 3. A compact, fully-convolutional network (no fixed input size)

Three `Conv-ReLU` blocks, each followed by `MaxPool2d(2, 2)` (2×
downsampling), then a **1×1 convolution acting as the classifier** (instead
of `Flatten` + `Linear`) followed by **Global Average Pooling**. Because
the last two operations are shape-agnostic, this network runs on any input
spatial size — that's the property we're about to demonstrate.

In [7]:
class TinyFCN(nn.Module):
    """A compact, fully-convolutional network.

    - Downsamples 2x at a time with nn.MaxPool2d, like a classic CNN.
    - Ends with a 1x1 convolution (playing the role of the fully-connected
      classifier) followed by global average pooling, so the network
      accepts any input spatial size unchanged.
    """
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(OrderedDict([
            ('conv1', nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)),
            ('relu1', nn.ReLU(inplace=True)),
            ('conv2', nn.Conv2d(32, 32, kernel_size=3, padding=1)),
            ('relu2', nn.ReLU(inplace=True)),
            ('pool1', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample

            ('conv3', nn.Conv2d(32, 64, kernel_size=3, padding=1)),
            ('relu3', nn.ReLU(inplace=True)),
            ('conv4', nn.Conv2d(64, 64, kernel_size=3, padding=1)),
            ('relu4', nn.ReLU(inplace=True)),
            ('pool2', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample

            ('conv5', nn.Conv2d(64, 128, kernel_size=3, padding=1)),
            ('relu5', nn.ReLU(inplace=True)),
            ('pool3', nn.MaxPool2d(kernel_size=2, stride=2)),   # 2x downsample
        ]))
        self.classifier = nn.Sequential(OrderedDict([
            ('conv_1x1', nn.Conv2d(128, num_classes, kernel_size=1)),  # replaces FC
            ('gap',      nn.AdaptiveAvgPool2d(1)),                    # GAP
            ('flatten',  nn.Flatten()),
        ]))

    def forward(self, x):
        return self.classifier(self.features(x))

model = TinyFCN(in_channels=3, num_classes=10)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params:,}")

Total trainable parameters: 140,714


## 4. Table 1 — input size 224×224×3

In [8]:
summarize(model, input_size=(3, 224, 224), max_layers=None);

Layer                Type               Output Shape       Param #
------------------------------------------------------------------
features.conv1       Conv2d             (1, 32, 224, 224)  896
features.relu1       ReLU               (1, 32, 224, 224)  0
features.conv2       Conv2d             (1, 32, 224, 224)  9,248
features.relu2       ReLU               (1, 32, 224, 224)  0
features.pool1       MaxPool2d          (1, 32, 112, 112)  0
features.conv3       Conv2d             (1, 64, 112, 112)  18,496
features.relu3       ReLU               (1, 64, 112, 112)  0
features.conv4       Conv2d             (1, 64, 112, 112)  36,928
features.relu4       ReLU               (1, 64, 112, 112)  0
features.pool2       MaxPool2d          (1, 64, 56, 56)    0
features.conv5       Conv2d             (1, 128, 56, 56)   73,856
features.relu5       ReLU               (1, 128, 56, 56)   0
features.pool3       MaxPool2d          (1, 128, 28, 28)   0
classifier.conv_1x1  Conv2d             (1, 10, 28, 

## 5. Table 2 — input size 120×120×3

In [9]:
summarize(model, input_size=(3, 120, 120), max_layers=None);

Layer                Type               Output Shape       Param #
------------------------------------------------------------------
features.conv1       Conv2d             (1, 32, 120, 120)  896
features.relu1       ReLU               (1, 32, 120, 120)  0
features.conv2       Conv2d             (1, 32, 120, 120)  9,248
features.relu2       ReLU               (1, 32, 120, 120)  0
features.pool1       MaxPool2d          (1, 32, 60, 60)    0
features.conv3       Conv2d             (1, 64, 60, 60)    18,496
features.relu3       ReLU               (1, 64, 60, 60)    0
features.conv4       Conv2d             (1, 64, 60, 60)    36,928
features.relu4       ReLU               (1, 64, 60, 60)    0
features.pool2       MaxPool2d          (1, 64, 30, 30)    0
features.conv5       Conv2d             (1, 128, 30, 30)   73,856
features.relu5       ReLU               (1, 128, 30, 30)   0
features.pool3       MaxPool2d          (1, 128, 15, 15)   0
classifier.conv_1x1  Conv2d             (1, 10, 15, 

## 6. The point to take from the two tables

Look at the last three rows of each table (`conv_1x1`, `gap`, `flatten`):
the spatial size differs between the two inputs right up until the GAP
layer, but the **final output shape and the total parameter count are
identical** for both input sizes. That is exactly the property the lecture
motivates in the "1×1 convolutions" section: replacing the fully-connected
classifier with a 1×1 convolution + GAP removes the fixed-input-size
constraint that a `Flatten` + `Linear` classifier would otherwise impose.

In [10]:
print(f"Output shape @224: {model(torch.zeros(1, 3, 224, 224)).shape}")
print(f"Output shape @120: {model(torch.zeros(1, 3, 120, 120)).shape}")

Output shape @224: torch.Size([1, 10])
Output shape @120: torch.Size([1, 10])
